In [1]:
import gdown

# Ganti dengan ID asli dari Google Drive
image_id = "157f9aE3ZhRSdIuIbP2PRG8ub9JJWvMGk"
mask_id = "1d08fFpEvK4D6YTKfRlNuv_OlIxigZxl6"

# Download image.zip
gdown.download(f"https://drive.google.com/uc?id={image_id}", "image.zip", quiet=False)

# Download mask.zip
gdown.download(f"https://drive.google.com/uc?id={mask_id}", "mask.zip", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=157f9aE3ZhRSdIuIbP2PRG8ub9JJWvMGk
From (redirected): https://drive.google.com/uc?id=157f9aE3ZhRSdIuIbP2PRG8ub9JJWvMGk&confirm=t&uuid=abb5fe5b-b1d3-4ca3-b328-47772d431847
To: /kaggle/working/image.zip
100%|██████████| 569M/569M [00:05<00:00, 106MB/s]  
Downloading...
From: https://drive.google.com/uc?id=1d08fFpEvK4D6YTKfRlNuv_OlIxigZxl6
To: /kaggle/working/mask.zip
100%|██████████| 5.66M/5.66M [00:00<00:00, 146MB/s]


'mask.zip'

In [2]:
import os
import zipfile

# Target folder
base_dir = "/kaggle/working/dataset"
image_dir = os.path.join(base_dir, "image")
mask_dir = os.path.join(base_dir, "mask")

# Extract image.zip
with zipfile.ZipFile("image.zip", 'r') as zip_ref:
    zip_ref.extractall(base_dir)

# Extract mask.zip
with zipfile.ZipFile("mask.zip", 'r') as zip_ref:
    zip_ref.extractall(base_dir)

print("Ekstraksi selesai!")

Ekstraksi selesai!


In [3]:
image_dir

'/kaggle/working/dataset/image'

In [4]:
%%writefile clean_dataset_layout.py
#!/usr/bin/env python3
"""
Merapikan dataset: dari dataset/image/<id>/000.png
menjadi clean_dataset/image/<id>_000.png (idem untuk mask).

Secara default membaca ./dataset dan menulis ke ./clean_dataset (salinan file).
"""

from __future__ import annotations

import argparse
import shutil
import sys
from pathlib import Path


def collect_pairs(root: Path, sub: str) -> list[tuple[Path, str, str]]:
    """
    root/sub/<patient_id>/<file> -> daftar (src_path, patient_id, filename).
    """
    base = root / sub
    if not base.is_dir():
        return []
    out: list[tuple[Path, str, str]] = []
    for patient_dir in sorted(p for p in base.iterdir() if p.is_dir()):
        pid = patient_dir.name
        for f in sorted(patient_dir.iterdir()):
            if f.is_file() and f.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}:
                out.append((f, pid, f.name))
    return out


def main() -> int:
    ap = argparse.ArgumentParser(
        description="Flatten dataset/image|mask/<id>/slice.ext -> clean_dataset/.../<id>_slice.ext"
    )
    ap.add_argument(
        "--source",
        type=Path,
        default=Path(__file__).resolve().parent / "dataset",
        help="Folder berisi image/ dan mask/ (default: ./dataset)",
    )
    ap.add_argument(
        "--dest",
        type=Path,
        default=Path(__file__).resolve().parent / "clean_dataset",
        help="Output root (default: ./clean_dataset)",
    )
    ap.add_argument(
        "--symlink",
        action="store_true",
        help="Buat symlink ke file asli (hemat ruang); default: salin file",
    )
    ap.add_argument(
        "--dry-run",
        action="store_true",
        help="Hanya cetak rencana, tidak menulis",
    )
    args = ap.parse_args()
    src_root: Path = args.source.resolve()
    dst_root: Path = args.dest.resolve()

    if not src_root.is_dir():
        print(f"Sumber tidak ada atau bukan folder: {src_root}", file=sys.stderr)
        return 1

    for kind in ("image", "mask"):
        pairs = collect_pairs(src_root, kind)
        if not pairs:
            print(f"[skip] Tidak ada file di {src_root / kind}")
            continue
        out_dir = dst_root / kind
        if not args.dry_run:
            out_dir.mkdir(parents=True, exist_ok=True)

        for src, pid, fname in pairs:
            new_name = f"{pid}_{fname}"
            dst = out_dir / new_name
            if args.dry_run:
                print(f"{src} -> {dst}")
                continue
            if dst.exists() or dst.is_symlink():
                # Hindari timpa tanpa sadar; bisa diubah ke overwrite jika perlu
                print(f"[skip exists] {dst}", file=sys.stderr)
                continue
            if args.symlink:
                dst.symlink_to(src.resolve(), target_is_directory=False)
            else:
                shutil.copy2(src, dst)

        print(f"[{kind}] {len(pairs)} file -> {out_dir}")

    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except BrokenPipeError:
        # stdout ditutup lebih awal oleh pembaca (mis. `| head`); bukan error skrip
        raise SystemExit(0)


Writing clean_dataset_layout.py


In [5]:
!python clean_dataset_layout.py

[image] 10972 file -> /kaggle/working/clean_dataset/image
[mask] 10972 file -> /kaggle/working/clean_dataset/mask


In [6]:
%%writefile requirements.txt
Pillow

# U-Net segmentasi (train_unet.py)
torch
torchvision
segmentation-models-pytorch
albumentations
tqdm


Writing requirements.txt


In [7]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 1.5 MB/s eta 0:00:00a 0:00:01


In [8]:
!mkdir unet_segmentation

In [9]:
%%writefile unet_segmentation/dataset.py
"""Dataset slice dari clean_dataset (image/ + mask/ datar, nama sama)."""

from __future__ import annotations

import random
from pathlib import Path

import albumentations as A
import numpy as np
import torch
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset


def patient_id_from_stem(stem: str) -> str:
    return stem.rsplit("_", 1)[0]


def list_patients_and_files(clean_root: Path) -> tuple[list[str], list[str]]:
    img_dir = clean_root / "image"
    if not img_dir.is_dir():
        raise FileNotFoundError(f"Tidak ada folder image: {img_dir}")
    stems: list[str] = []
    patients: set[str] = set()
    for p in sorted(img_dir.glob("*.png")):
        stem = p.stem
        m = clean_root / "mask" / f"{stem}.png"
        if not m.is_file():
            continue
        stems.append(stem)
        patients.add(patient_id_from_stem(stem))
    return sorted(patients), stems


def split_by_patient(
    stems: list[str],
    val_ratio: float,
    seed: int,
) -> tuple[list[str], list[str]]:
    by_p: dict[str, list[str]] = {}
    for s in stems:
        pid = patient_id_from_stem(s)
        by_p.setdefault(pid, []).append(s)
    pids = list(by_p.keys())
    rng = random.Random(seed)
    rng.shuffle(pids)
    n_val = max(1, int(round(len(pids) * val_ratio)))
    if n_val >= len(pids):
        n_val = max(1, len(pids) - 1)
    val_p = set(pids[:n_val])
    train_stems = [s for s in stems if patient_id_from_stem(s) not in val_p]
    val_stems = [s for s in stems if patient_id_from_stem(s) in val_p]
    return train_stems, val_stems


def build_transforms(train: bool, image_size: int | None = None) -> A.Compose:
    """Augmentasi spasial + kontras pada image; mask ikut flip/resize saja."""
    ops: list = []
    if train:
        ops.extend(
            [
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
                A.GaussNoise(var_limit=(5.0, 40.0), p=0.2),
            ]
        )
    if image_size is not None:
        ops.append(A.Resize(image_size, image_size))
    ops.extend(
        [
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]
    )
    return A.Compose(ops)


class CleanDataset(Dataset):
    def __init__(
        self,
        clean_root: Path,
        stems: list[str],
        train: bool,
        image_size: int | None = None,
    ) -> None:
        self.root = Path(clean_root)
        self.stems = stems
        self.train = train
        self.tf = build_transforms(train=train, image_size=image_size)

    def __len__(self) -> int:
        return len(self.stems)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        stem = self.stems[idx]
        ip = self.root / "image" / f"{stem}.png"
        mp = self.root / "mask" / f"{stem}.png"
        img = np.array(Image.open(ip).convert("RGB"), dtype=np.float32)
        mask = np.array(Image.open(mp).convert("L"), dtype=np.float32)
        mask_u8 = (mask > 127.0).astype(np.uint8)
        aug = self.tf(image=img, mask=mask_u8)
        img_t = aug["image"]
        m = aug["mask"]
        if m.ndim == 2:
            m = m.unsqueeze(0)
        mask_bin = (m.float() > 0.5).float()
        return {"image": img_t, "mask": mask_bin, "stem": stem}


Writing unet_segmentation/dataset.py


In [10]:
%%writefile unet_segmentation/metrics.py
"""Metrik evaluasi segmentasi biner (per-slice dan agregat piksel)."""

from __future__ import annotations

import torch


def _flatten_batch(x: torch.Tensor) -> torch.Tensor:
    return x.view(x.shape[0], -1)


@torch.no_grad()
def dice_per_slice(pred_prob: torch.Tensor, mask: torch.Tensor, thr: float = 0.5) -> torch.Tensor:
    """
    Dice per sampel (slice) shape (B,1,H,W), nilai dalam [0,1].
    Jika GT kosong: Dice=1 jika pred kosong, else 0.
    """
    pred = (pred_prob > thr).float()
    gt = (mask > 0.5).float()
    b = pred.shape[0]
    eps = 1e-6
    out = pred.new_zeros(b)
    pf = _flatten_batch(pred)
    gf = _flatten_batch(gt)
    for i in range(b):
        p, g = pf[i], gf[i]
        inter = (p * g).sum()
        gsum, psum = g.sum(), p.sum()
        if gsum <= eps:
            out[i] = 1.0 if psum <= eps else 0.0
        else:
            out[i] = (2.0 * inter + eps) / (psum + gsum + eps)
    return out


@torch.no_grad()
def iou_per_slice(pred_prob: torch.Tensor, mask: torch.Tensor, thr: float = 0.5) -> torch.Tensor:
    pred = (pred_prob > thr).float()
    gt = (mask > 0.5).float()
    b = pred.shape[0]
    eps = 1e-6
    out = pred.new_zeros(b)
    pf = _flatten_batch(pred)
    gf = _flatten_batch(gt)
    for i in range(b):
        p, g = pf[i], gf[i]
        inter = (p * g).sum()
        union = p.sum() + g.sum() - inter
        if union <= eps:
            out[i] = 1.0 if inter <= eps else 0.0
        else:
            out[i] = (inter + eps) / (union + eps)
    return out


@torch.no_grad()
def precision_recall_per_slice(
    pred_prob: torch.Tensor, mask: torch.Tensor, thr: float = 0.5
) -> tuple[torch.Tensor, torch.Tensor]:
    pred = (pred_prob > thr).float()
    gt = (mask > 0.5).float()
    b = pred.shape[0]
    eps = 1e-6
    prec = pred.new_zeros(b)
    rec = pred.new_zeros(b)
    pf = _flatten_batch(pred)
    gf = _flatten_batch(gt)
    for i in range(b):
        p, g = pf[i], gf[i]
        tp = (p * g).sum()
        fp = (p * (1 - g)).sum()
        fn = ((1 - p) * g).sum()
        prec[i] = (tp + eps) / (tp + fp + eps)
        rec[i] = (tp + eps) / (tp + fn + eps)
    return prec, rec


@torch.no_grad()
def pixel_accuracy(pred_prob: torch.Tensor, mask: torch.Tensor, thr: float = 0.5) -> float:
    pred = (pred_prob > thr).float()
    gt = (mask > 0.5).float()
    return float((pred == gt).float().mean())


@torch.no_grad()
def global_dice_from_confusion(pred_prob: torch.Tensor, mask: torch.Tensor, thr: float = 0.5) -> float:
    """Satu Dice dari total TP, FP, FN di seluruh batch (micro / voxel-level)."""
    pred = (pred_prob > thr).float()
    gt = (mask > 0.5).float()
    inter = (pred * gt).sum()
    eps = 1e-6
    return float((2.0 * inter + eps) / (pred.sum() + gt.sum() + eps))


@torch.no_grad()
def global_specificity(pred_prob: torch.Tensor, mask: torch.Tensor, thr: float = 0.5) -> float:
    pred = (pred_prob > thr).float()
    gt = (mask > 0.5).float()
    tn = ((1 - pred) * (1 - gt)).sum()
    fp = (pred * (1 - gt)).sum()
    eps = 1e-6
    return float((tn + eps) / (tn + fp + eps))


Writing unet_segmentation/metrics.py


In [11]:
%%writefile unet_segmentation/train.py
"""
Training U-Net (PyTorch) dengan encoder ImageNet-pretrained (segmentation_models_pytorch).
Dataset: clean_dataset/image + clean_dataset/mask.
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import segmentation_models_pytorch as smp
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from unet_segmentation.dataset import CleanDataset, list_patients_and_files, split_by_patient
from unet_segmentation.metrics import (
    dice_per_slice,
    iou_per_slice,
    precision_recall_per_slice,
)


SLICE_DICE_TARGET = 0.76


def build_model(encoder: str, pretrained: bool) -> nn.Module:
    weights = "imagenet" if pretrained else None
    return smp.Unet(
        encoder_name=encoder,
        encoder_weights=weights,
        in_channels=3,
        classes=1,
        activation=None,
    )


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> dict[str, float]:
    model.eval()
    all_slice_dice: list[torch.Tensor] = []
    all_slice_iou: list[torch.Tensor] = []
    all_prec: list[torch.Tensor] = []
    all_rec: list[torch.Tensor] = []
    inter = torch.zeros(1, device=device)
    pred_sum = torch.zeros(1, device=device)
    gt_sum = torch.zeros(1, device=device)
    tn_acc = torch.zeros(1, device=device)
    fp_acc = torch.zeros(1, device=device)
    pix_correct = 0
    pix_total = 0
    eps = 1e-6

    for batch in loader:
        imgs = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        logits = model(imgs)
        prob = torch.sigmoid(logits)

        d = dice_per_slice(prob, masks)
        j = iou_per_slice(prob, masks)
        pr, rc = precision_recall_per_slice(prob, masks)
        all_slice_dice.append(d.cpu())
        all_slice_iou.append(j.cpu())
        all_prec.append(pr.cpu())
        all_rec.append(rc.cpu())

        pred_bin = (prob > 0.5).float()
        gt = (masks > 0.5).float()
        inter += (pred_bin * gt).sum()
        pred_sum += pred_bin.sum()
        gt_sum += gt.sum()
        tn_acc += ((1.0 - pred_bin) * (1.0 - gt)).sum()
        fp_acc += (pred_bin * (1.0 - gt)).sum()
        pix_correct += int((pred_bin == gt).sum().item())
        pix_total += int(gt.numel())

    slice_dice = torch.cat(all_slice_dice).mean().item()
    slice_iou = torch.cat(all_slice_iou).mean().item()
    slice_precision = torch.cat(all_prec).mean().item()
    slice_recall = torch.cat(all_rec).mean().item()
    f1 = (
        2 * slice_precision * slice_recall / (slice_precision + slice_recall + 1e-8)
        if (slice_precision + slice_recall) > 0
        else 0.0
    )

    global_dice = float((2.0 * inter + eps) / (pred_sum + gt_sum + eps))
    global_spec = float((tn_acc + eps) / (tn_acc + fp_acc + eps))
    mean_pix = float((pix_correct + eps) / (pix_total + eps)) if pix_total else 0.0

    return {
        "slice_level_dice": slice_dice,
        "slice_level_iou": slice_iou,
        "slice_level_precision": slice_precision,
        "slice_level_recall": slice_recall,
        "slice_level_f1": float(f1),
        "mean_pixel_accuracy": mean_pix,
        "global_dice_micro": global_dice,
        "global_specificity": global_spec,
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler | None,
    dice_loss: nn.Module,
    bce_loss: nn.Module,
    device: torch.device,
    use_amp: bool,
) -> float:
    model.train()
    running = 0.0
    n = 0
    amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
    for batch in tqdm(loader, desc="train", leave=False):
        imgs = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp and scaler is not None:
            with torch.autocast(device_type=device.type, dtype=amp_dtype):
                logits = model(imgs)
                loss = dice_loss(logits, masks) + bce_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss = dice_loss(logits, masks) + bce_loss(logits, masks)
            loss.backward()
            optimizer.step()
        running += float(loss.detach())
        n += 1
    return running / max(n, 1)


def main() -> int:
    ap = argparse.ArgumentParser(description="U-Net segmentasi (clean_dataset)")
    root = Path(__file__).resolve().parent.parent
    ap.add_argument(
        "--clean-root",
        type=Path,
        default=root / "clean_dataset",
        help="Root clean_dataset berisi image/ dan mask/",
    )
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--val-ratio", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--encoder", type=str, default="resnet34", help="Encoder smp, mis. resnet34, efficientnet-b0")
    ap.add_argument("--no-pretrained", action="store_true", help="Tanpa bobot ImageNet pada encoder")
    ap.add_argument("--image-size", type=int, default=None, help="Resize sisi (default: ukuran asli)")
    ap.add_argument("--out-dir", type=Path, default=root / "checkpoints_unet")
    ap.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--no-amp", action="store_true")
    args = ap.parse_args()

    clean_root = args.clean_root.resolve()
    if not clean_root.is_dir():
        print(f"clean_dataset tidak ditemukan: {clean_root}", file=sys.stderr)
        return 1

    torch.manual_seed(args.seed)
    device = torch.device(args.device)
    use_amp = device.type == "cuda" and not args.no_amp
    scaler = None
    if use_amp:
        try:
            scaler = torch.amp.GradScaler("cuda")
        except (TypeError, AttributeError):
            scaler = torch.cuda.amp.GradScaler()

    _, stems = list_patients_and_files(clean_root)
    if len(stems) < 10:
        print(f"Terlalu sedikit sampel berpasangan: {len(stems)}", file=sys.stderr)
        return 1

    train_stems, val_stems = split_by_patient(stems, args.val_ratio, args.seed)
    print(f"Sampel train: {len(train_stems)}, val: {len(val_stems)} (split per pasien)")

    train_ds = CleanDataset(clean_root, train_stems, train=True, image_size=args.image_size)
    val_ds = CleanDataset(clean_root, val_stems, train=False, image_size=args.image_size)
    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        pin_memory=device.type == "cuda",
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=device.type == "cuda",
    )

    model = build_model(args.encoder, pretrained=not args.no_pretrained)
    model.to(device)

    dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)
    bce_loss = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(args.epochs, 1))

    args.out_dir.mkdir(parents=True, exist_ok=True)
    best_dice = -1.0
    best_path = args.out_dir / "best_unet.pt"

    for epoch in range(1, args.epochs + 1):
        loss_tr = train_one_epoch(
            model, train_loader, optimizer, scaler, dice_loss, bce_loss, device, use_amp
        )
        metrics = evaluate(model, val_loader, device)
        sched.step()

        print(
            f"Epoch {epoch}/{args.epochs}  train_loss={loss_tr:.4f}  "
            f"slice_level_dice={metrics['slice_level_dice']:.4f}  "
            f"slice_level_iou={metrics['slice_level_iou']:.4f}  "
            f"slice_prec={metrics['slice_level_precision']:.4f}  "
            f"slice_rec={metrics['slice_level_recall']:.4f}  "
            f"slice_f1={metrics['slice_level_f1']:.4f}  "
            f"pix_acc={metrics['mean_pixel_accuracy']:.4f}  "
            f"global_dice={metrics['global_dice_micro']:.4f}  "
            f"spec={metrics['global_specificity']:.4f}"
        )

        sd = metrics["slice_level_dice"]
        if sd > SLICE_DICE_TARGET:
            print(
                f">>> slice_level_dice ({sd:.4f}) > target ({SLICE_DICE_TARGET}): "
                "Sudah memenuhi target evaluasi."
            )

        if sd > best_dice:
            best_dice = sd
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "encoder": args.encoder,
                    "metrics": metrics,
                },
                best_path,
            )
            print(f"  (simpan checkpoint terbaik -> {best_path})")

    print(f"Selesai. slice_level_dice terbaik validasi: {best_dice:.4f}")
    if best_dice > SLICE_DICE_TARGET:
        print(
            f"Model terbaik (slice_level_dice={best_dice:.4f}) di atas target {SLICE_DICE_TARGET}."
        )
    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except BrokenPipeError:
        raise SystemExit(0)


Writing unet_segmentation/train.py


In [12]:
%%writefile unet_segmentation/__init__.py
"""U-Net segmentasi stroke dari clean_dataset."""

Writing unet_segmentation/__init__.py


In [13]:
%%writefile train_unet.py
#!/usr/bin/env python3
"""Entry point: jalankan dari root proyek, mis. `python train_unet.py`."""

from unet_segmentation.train import main

if __name__ == "__main__":
    raise SystemExit(main())

Writing train_unet.py


In [14]:
!python train_unet.py --clean-root /kaggle/working/clean_dataset --epochs 1 --batch-size 8

Sampel train: 9355, val: 1617 (split per pasien)
/kaggle/working/unet_segmentation/dataset.py:65: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 40.0), p=0.2),
config.json: 100%|██████████████████████████████| 156/156 [00:00<00:00, 737kB/s]
model.safetensors: 100%|███████████████████| 87.3M/87.3M [00:01<00:00, 59.2MB/s]
Epoch 1/1  train_loss=0.1506  slice_level_dice=0.7669  slice_level_iou=0.7669  slice_prec=0.7669  slice_rec=1.0000  slice_f1=0.8680  pix_acc=0.9997  global_dice=0.0000  spec=0.9997
>>> slice_level_dice (0.7669) > target (0.76): Sudah memenuhi target evaluasi.
  (simpan checkpoint terbaik -> /kaggle/working/checkpoints_unet/best_unet.pt)
Selesai. slice_level_dice terbaik validasi: 0.7669
Model terbaik (slice_level_dice=0.7669) di atas target 0.76.


In [15]:
import shutil
import os

src = "/kaggle/working/checkpoints_unet/best_unet.pt"
dst_dir = "/kaggle/working/output"

os.makedirs(dst_dir, exist_ok=True)

dst = os.path.join(dst_dir, "best_unet.pt")
shutil.copy(src, dst)

print("File siap didownload di panel Output:", dst)

File siap didownload di panel Output: /kaggle/working/output/best_unet.pt
